# Results Overview — Atomic Extraction

Aggregates all submitted results from `results/register.csv` for the extraction task (atomic variables).
Each model submits one row per variable. Metrics vary by variable type:
- **Year fields** (`birth_year`, `edu_start`, `edu_end`): accuracy (exact match) + MAE
- **Place/country** (`birth_place`, `birth_country`): char similarity (headline) + accuracy when predicted
- **Coded fields** (`sex`, `edu_degree`): accuracy + accuracy when predicted

## 1. Imports & config

In [ ]:
import json as _json
from pathlib import Path

import matplotlib.colors as mcolors
import matplotlib.patches as mpatches
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

plt.rcParams["figure.facecolor"] = "white"
plt.rcParams["axes.facecolor"]   = "white"

REGISTER = Path("/home/tom/projects/corex_eval/results/register.csv")
TASK     = "extraction"

ATOMIC_VARS = ["birth_year", "birth_place", "birth_country", "sex", "edu_start", "edu_end", "edu_degree"]
YEAR_VARS   = ["birth_year", "edu_start", "edu_end"]
CHAR_VARS   = ["birth_place", "birth_country"]
CODED_VARS  = ["sex", "edu_degree"]

VAR_LABELS = {
    "birth_year":    "Birth year",
    "birth_place":   "Birth place",
    "birth_country": "Birth country",
    "sex":           "Sex",
    "edu_start":     "Edu. start",
    "edu_end":       "Edu. end",
    "edu_degree":    "Edu. degree",
}

FAMILY_COLORS = {
    "OSS LLM": "#2196F3",
    "Closed":  "#FF5722",
}

## 2. Model catalogue

In [ ]:
# (family, display_name, experiment_folder)
CATALOGUE = [
    ("OSS LLM", "Llama 3.3 70B",  "lmstudio_llama33_70b_atomic"),
    ("OSS LLM", "Qwen3 80B",       "lmstudio_qwen3_80b_atomic"),
    ("Closed",  "Claude Opus 4.6", "claude_opus46_atomic"),
    ("Closed",  "GPT-5.3",         "gpt53_atomic"),
]

## 3. Load register

In [ ]:
reg = pd.read_csv(REGISTER)
reg = reg[reg["task"] == TASK].copy()
print(f"Total extraction rows: {len(reg)}")

def _folder(path):
    parts = str(path).replace("\\", "/").split("/")
    return parts[-2] if len(parts) >= 2 else str(path)

reg["exp_folder"] = reg["experiment_path"].apply(_folder)
reg["accuracy"]   = pd.to_numeric(reg["accuracy"], errors="coerce")

# Best row per (folder, variable)
reg_best = (
    reg[reg["accuracy"].notna()]
    .sort_values("accuracy", ascending=False)
    .drop_duplicates(subset=["exp_folder", "variable"], keep="first")
    .set_index(["exp_folder", "variable"])
)
print(f"Valid rows: {len(reg_best)}")
print("Folders found:", sorted(reg_best.index.get_level_values(0).unique().tolist()))

## 4. Build results table

In [ ]:
def get_metric(folder, variable):
    key = (folder, variable)
    if key not in reg_best.index:
        return None, None, None
    row = reg_best.loc[key]
    acc = float(row["accuracy"]) if pd.notna(row.get("accuracy")) else None
    mae = float(row["mae"])      if pd.notna(row.get("mae"))      else None
    awp = None
    try:
        mj  = _json.loads(str(row["metrics_json"]))
        awp = mj.get("accuracy_when_predicted")
    except Exception:
        pass
    return acc, awp, mae

records = []
for family, name, folder in CATALOGUE:
    row = {"Family": family, "Model": name}
    for v in ATOMIC_VARS:
        acc, awp, mae = get_metric(folder, v)
        label = VAR_LABELS[v]
        if v in YEAR_VARS:
            row[label]           = f"{acc:.3f}" if acc is not None else "TBD"
            row[label + " MAE"]  = f"{mae:.2f}"  if mae is not None else "TBD"
        else:
            row[label]           = f"{acc:.3f}" if acc is not None else "TBD"
            row[label + " (pred)"] = f"{awp:.3f}" if awp is not None else "TBD"
    records.append(row)

results_df = pd.DataFrame(records)
results_df.style.hide(axis="index")

## 5. Heatmap — accuracy by model × variable

In [ ]:
folders     = [f for _, _, f in CATALOGUE]
model_names = [n for _, n, _ in CATALOGUE]

matrix = np.full((len(CATALOGUE), len(ATOMIC_VARS)), np.nan)
for i, folder in enumerate(folders):
    for j, v in enumerate(ATOMIC_VARS):
        acc, _, _ = get_metric(folder, v)
        if acc is not None:
            matrix[i, j] = acc

fig, ax = plt.subplots(figsize=(12, 4))
im = ax.imshow(matrix, cmap="RdYlGn", vmin=0, vmax=1, aspect="auto")
plt.colorbar(im, ax=ax, fraction=0.03, pad=0.02, label="Accuracy / Char similarity")

ax.set_xticks(range(len(ATOMIC_VARS)))
ax.set_xticklabels([VAR_LABELS[v] for v in ATOMIC_VARS], fontsize=10)
ax.set_yticks(range(len(CATALOGUE)))
ax.set_yticklabels(model_names, fontsize=10)
ax.set_title("Atomic extraction — headline accuracy per model x variable",
             fontsize=12, fontweight="bold")

for i in range(len(CATALOGUE)):
    for j in range(len(ATOMIC_VARS)):
        v = matrix[i, j]
        if not np.isnan(v):
            ax.text(j, i, f"{v:.3f}", ha="center", va="center",
                    fontsize=9, color="black" if 0.3 < v < 0.8 else "white")
plt.tight_layout()
plt.show()

## 6. Bar chart — per-variable comparison across models

In [ ]:
x       = np.arange(len(ATOMIC_VARS))
n       = len(CATALOGUE)
w       = 0.18
offsets = np.linspace(-(n-1)/2*w, (n-1)/2*w, n)

fig, ax = plt.subplots(figsize=(14, 5))
for i, (family, name, folder) in enumerate(CATALOGUE):
    vals   = [get_metric(folder, v)[0] or 0 for v in ATOMIC_VARS]
    color  = FAMILY_COLORS[family]
    alpha  = 0.65 + 0.2 * (i % 2)
    ax.bar(x + offsets[i], vals, w, label=name, color=color,
           alpha=alpha, edgecolor="white")

ax.set_xticks(x)
ax.set_xticklabels([VAR_LABELS[v] for v in ATOMIC_VARS], fontsize=10)
ax.set_ylabel("Accuracy / Char similarity")
ax.set_ylim(0, 1.05)
ax.legend(fontsize=9, ncol=2)
ax.set_title("Atomic extraction — model comparison per variable",
             fontsize=12, fontweight="bold")
plt.tight_layout()
plt.show()

## 7. Per-country accuracy (select model & variable)

In [ ]:
MODEL_FOLDER = "gpt53_atomic"   # change to any folder in catalogue
VARIABLE     = "edu_degree"  # change to any atomic variable

def get_per_country(folder, variable):
    key = (folder, variable)
    if key not in reg_best.index:
        print(f"No result for ({folder}, {variable})")
        return {}
    try:
        return _json.loads(str(reg_best.loc[key, "metrics_json"])).get("per_country", {})
    except Exception as e:
        print(f"JSON parse error: {e}")
        return {}

pc = get_per_country(MODEL_FOLDER, VARIABLE)
if not pc:
    print("No per-country data found.")
else:
    pc_df = pd.DataFrame([
        {"country": k, "accuracy": v["accuracy"], "n": v["n_evaluated"]}
        for k, v in pc.items() if v.get("n_evaluated", 0) > 0
    ]).sort_values("accuracy", ascending=True)

    fig, ax = plt.subplots(figsize=(10, max(6, len(pc_df) * 0.35)))
    colors = plt.cm.RdYlGn(pc_df["accuracy"].fillna(0).values)
    bars = ax.barh(pc_df["country"], pc_df["accuracy"].fillna(0), color=colors, edgecolor="white")
    for bar, n in zip(bars, pc_df["n"]):
        ax.text(bar.get_width() + 0.005, bar.get_y() + bar.get_height()/2,
                f"n={n}", va="center", fontsize=8)
    ax.set_xlim(0, 1.15)
    ax.set_xlabel("Accuracy / Char similarity")
    ax.set_title(f"Per-country — {MODEL_FOLDER} / {VARIABLE}", fontweight="bold")
    plt.tight_layout()
    plt.show()

## 8. LaTeX table export

In [ ]:
def to_latex(df, caption, label):
    cols = [c for c in df.columns if c not in ("Family",)]
    d = df[cols].copy()
    col_str = " & ".join(c.replace("_", "\\_") for c in d.columns) + " \\\\"
    rows_str = []
    prev_family = None
    for _, row in df.iterrows():
        if row["Family"] != prev_family and prev_family is not None:
            rows_str.append("\\midrule")
        prev_family = row["Family"]
        rows_str.append(" & ".join(str(row[c]) if pd.notna(row[c]) else "TBD" for c in cols) + " \\\\")
    body = "\n    ".join(rows_str)
    n_cols = len(cols)
    return (
        "\\begin{table}[ht]\n"
        "\\centering\n"
        f"\\caption{{{caption}}}\n"
        f"\\label{{{label}}}\n"
        f"\\begin{{tabular}}{{l{'c' * (n_cols - 1)}}}\n"
        "\\toprule\n"
        f"{col_str}\n"
        "\\midrule\n"
        f"    {body}\n"
        "\\bottomrule\n"
        "\\end{tabular}\n"
        "\\end{table}"
    )

print(to_latex(results_df, "Atomic extraction results", "tab:extraction-atomic"))